# Assignment 3b: Advanced Gradio RAG Frontend
## Day 6 Session 2 - Building Configurable RAG Applications

In this assignment, you'll extend your basic RAG interface with advanced configuration options to create a professional, feature-rich RAG application.

**New Features to Add:**
- Model selection dropdown (gpt-4o, gpt-4o-mini)
- Temperature slider (0 to 1 with 0.1 intervals)
- Chunk size configuration
- Chunk overlap configuration  
- Similarity top-k slider
- Node postprocessor multiselect
- Similarity cutoff slider
- Response synthesizer multiselect

**Learning Objectives:**
- Advanced Gradio components and interactions
- Dynamic RAG configuration
- Professional UI design patterns
- Parameter validation and handling
- Building production-ready AI applications

**Prerequisites:**
- Completed Assignment 3a (Basic Gradio RAG)
- Understanding of RAG parameters and their effects


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install -q -r "/content/drive/MyDrive/ColabNotebooks/python-project/ai-accelerator-C5/Day_6/session_2/requirements.txt" # your path

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 11.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 66.8 MB/s eta 

In [5]:
import os
from getpass import getpass

# securely input your key
os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter key")
print("✓ OpenRouter key set successfully")



Enter your OpenRouter key··········
✓ OpenRouter key set successfully


## 📚 Part 1: Setup and Imports

Import all necessary libraries including advanced RAG components for configuration options.

**Note:** This assignment uses OpenRouter for LLM access (not OpenAI). Make sure you have your `OPENROUTER_API_KEY` environment variable set.


In [6]:
# Import all required libraries
import gradio as gr
import os
from pathlib import Path
from typing import Dict, List, Optional, Any

# LlamaIndex core components
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext, Settings
from llama_index.vector_stores.lancedb import LanceDBVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openrouter import OpenRouter

# Advanced RAG components
from llama_index.core.postprocessor import SimilarityPostprocessor
from llama_index.core.response_synthesizers import TreeSummarize, Refine, CompactAndRefine
from llama_index.core.retrievers import VectorIndexRetriever

print("✅ All libraries imported successfully!")


✅ All libraries imported successfully!


## 🤖 Part 2: Advanced RAG Backend Class

Create an advanced RAG backend that supports dynamic configuration of all parameters.


In [10]:
class AdvancedRAGBackend:
    """Advanced RAG backend with configurable parameters and memory."""

    def __init__(self):
        self.index = None
        self.chat_history = []
        self.available_models = ["gpt-4o", "gpt-4o-mini"]
        self.available_postprocessors = ["SimilarityPostprocessor"]
        self.available_synthesizers = ["TreeSummarize", "Refine", "CompactAndRefine", "Default"]
        self.update_settings()

    def update_settings(self, model: str = "gpt-4o-mini", temperature: float = 0.1, chunk_size: int = 512, chunk_overlap: int = 50):
        api_key = os.getenv("OPENROUTER_API_KEY")
        if api_key:
            Settings.llm = OpenRouter(api_key=api_key, model=model, temperature=temperature)
        Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", trust_remote_code=True)
        Settings.chunk_size = chunk_size
        Settings.chunk_overlap = chunk_overlap

    def initialize_database(self, data_folder="/content/drive/MyDrive/ColabNotebooks/python-project/ai-accelerator-C5/Day_6/session_2/data"):
        if not Path(data_folder).exists(): return f"❌ Data folder '{data_folder}' not found!"
        try:
            vector_store = LanceDBVectorStore(uri="/content/storage/advanced_rag_vectordb", table_name="documents")
            reader = SimpleDirectoryReader(input_dir=data_folder, recursive=True)
            documents = reader.load_data()
            storage_context = StorageContext.from_defaults(vector_store=vector_store)
            self.index = VectorStoreIndex.from_documents(documents, storage_context=storage_context, show_progress=True)
            return f"✅ Database initialized successfully with {len(documents)} documents!"
        except Exception as e: return f"❌ Error: {str(e)}"

    def get_postprocessor(self, name, cutoff): return SimilarityPostprocessor(similarity_cutoff=cutoff) if name == "SimilarityPostprocessor" else None
    def get_synthesizer(self, name):
        synths = {"TreeSummarize": TreeSummarize, "Refine": Refine, "CompactAndRefine": CompactAndRefine}
        return synths[name]() if name in synths else None

    def advanced_query(self, question, model, temperature, chunk_size, chunk_overlap, similarity_top_k, postprocessor_names, similarity_cutoff, synthesizer_name):
        if self.index is None: return {"response": "❌ Initialize database first!", "history": self.chat_history, "config": {}}
        try:
            self.update_settings(model, temperature, chunk_size, chunk_overlap)
            postprocessors = [self.get_postprocessor(n, similarity_cutoff) for n in postprocessor_names if self.get_postprocessor(n, similarity_cutoff)]
            synthesizer = self.get_synthesizer(synthesizer_name)

            query_engine = self.index.as_query_engine(similarity_top_k=similarity_top_k, node_postprocessors=postprocessors, response_synthesizer=synthesizer)
            response = str(query_engine.query(question))

            self.chat_history.append([question, response])
            return {"response": response, "history": self.chat_history, "config": {"model": model, "temperature": temperature, "chunk_size": chunk_size, "chunk_overlap": chunk_overlap, "similarity_top_k": similarity_top_k, "postprocessors": postprocessor_names, "similarity_cutoff": similarity_cutoff, "synthesizer": synthesizer_name}}
        except Exception as e: return {"response": f"❌ Error: {str(e)}", "history": self.chat_history, "config": {}}

rag_backend = AdvancedRAGBackend()

## 🎨 Part 3: Advanced Gradio Interface

Create a sophisticated Gradio interface with all the configuration options specified:
1. Database initialization button
2. Search query input and button  
3. Model selection dropdown
4. Temperature slider
5. Chunk size and overlap inputs
6. Similarity top-k slider
7. Node postprocessor multiselect
8. Similarity cutoff slider
9. Response synthesizer multiselect


In [14]:
import gradio as gr

def create_advanced_rag_interface():
    custom_css = ".gradio-container { font-family: -apple-system, sans-serif !important; } .config-column { background: rgba(0,0,0,0.03); border-radius: 15px; padding: 15px; } .dark .config-column { background: rgba(255,255,255,0.05); }"

    def handle_advanced_query(*args):
        result = rag_backend.advanced_query(*args)
        config_md = f"**System Parameters:**\n- 🤖 Model: {result['config'].get('model')}\n- 🌡️ Temp: {result['config'].get('temperature')}"
        return result["response"], result["history"], config_md

    with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="ROBO ChatBot") as interface:
        gr.Markdown("# 🤖 ROBO ChatBot\n*Advanced Intelligence Interface*")

        with gr.Row():
            with gr.Column(scale=1, elem_classes="config-column"):
                init_btn = gr.Button("Initialize Database")
                status_out = gr.Textbox(label="Status", placeholder="Idle...")
                model_dd = gr.Dropdown(["gpt-4o", "gpt-4o-mini"], label="Model", value="gpt-4o-mini")
                temp_sld = gr.Slider(0, 1, value=0.1, step=0.1, label="Temperature")
                c_size = gr.Number(label="Chunk Size", value=512)
                c_ovr = gr.Number(label="Overlap", value=50)
                top_k = gr.Slider(1, 20, value=5, label="Top-K")
                post_p = gr.CheckboxGroup(["SimilarityPostprocessor"], label="Post-Processors")
                cut_off = gr.Slider(0, 1, value=0.3, label="Cutoff")
                synth_dd = gr.Dropdown(["TreeSummarize", "Refine", "CompactAndRefine", "Default"], label="Strategy", value="Default")

            with gr.Column(scale=2):
                query_in = gr.Textbox(label="Query", placeholder="Ask something...", lines=3)
                submit_btn = gr.Button("Generate", variant="primary")
                ans_out = gr.Markdown(label="Response")

                with gr.Accordion("📜 Chat History", open=False):
                    history_display = gr.Chatbot(label="Conversation History")

                with gr.Accordion("⚙️ Metadata", open=False):
                    config_out = gr.Markdown()

        init_btn.click(lambda: rag_backend.initialize_database(), outputs=[status_out])
        submit_btn.click(handle_advanced_query, inputs=[query_in, model_dd, temp_sld, c_size, c_ovr, top_k, post_p, cut_off, synth_dd], outputs=[ans_out, history_display, config_out])

    return interface

advanced_interface = create_advanced_rag_interface()

/tmp/ipykernel_12586/4266034966.py:11: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="ROBO ChatBot") as interface:
/tmp/ipykernel_12586/4266034966.py:11: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="ROBO ChatBot") as interface:
/tmp/ipykernel_12586/4266034966.py:33: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  history_display = gr.Chatbot(label="Conversation History")
/tmp/ipykernel_12586/

## 🚀 Part 4: Launch Your Advanced Application

Launch your advanced Gradio application and test all the configuration options!


In [13]:
print("🎉 Launching your Advanced RAG Assistant...")
print("🔗 Your application will open in a new browser tab!")
print("")
print("⚠️  Make sure your OPENROUTER_API_KEY environment variable is set!")
print("")
print("📋 Testing Instructions:")
print("1. Click 'Initialize Vector Database' button first")
print("2. Wait for success message")
print("3. Configure your RAG parameters:")
print("   - Choose model (gpt-4o, gpt-4o-mini)")
print("   - Adjust temperature (0.0 = deterministic, 1.0 = creative)")
print("   - Set chunk size and overlap")
print("   - Choose similarity top-k")
print("   - Select postprocessors and synthesizer")
print("4. Enter a question and click 'Ask Question'")
print("5. Review both the response and configuration used")
print("")
print("🧪 Experiments to try:")
print("- Compare different models with the same question")
print("- Test temperature effects (0.1 vs 0.9)")
print("- Try different chunk sizes (256 vs 1024)")
print("- Compare synthesizers (TreeSummarize vs Refine)")
print("- Adjust similarity cutoff to filter results")

# Your code here:
advanced_interface.launch()

🎉 Launching your Advanced RAG Assistant...
🔗 Your application will open in a new browser tab!

⚠️  Make sure your OPENROUTER_API_KEY environment variable is set!

📋 Testing Instructions:
1. Click 'Initialize Vector Database' button first
2. Wait for success message
3. Configure your RAG parameters:
   - Choose model (gpt-4o, gpt-4o-mini)
   - Adjust temperature (0.0 = deterministic, 1.0 = creative)
   - Set chunk size and overlap
   - Choose similarity top-k
   - Select postprocessors and synthesizer
4. Enter a question and click 'Ask Question'
5. Review both the response and configuration used

🧪 Experiments to try:
- Compare different models with the same question
- Test temperature effects (0.1 vs 0.9)
- Try different chunk sizes (256 vs 1024)
- Compare synthesizers (TreeSummarize vs Refine)
- Adjust similarity cutoff to filter results
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this

## 💡 Understanding the Configuration Options

### Model Selection
- **gpt-4o**: Latest and most capable model, best quality responses
- **gpt-4o-mini**: Faster and cheaper while maintaining good quality

### Temperature (0.0 - 1.0)
- **0.0-0.3**: Deterministic, factual responses
- **0.4-0.7**: Balanced creativity and accuracy
- **0.8-1.0**: More creative and varied responses

### Chunk Size & Overlap
- **Chunk Size**: How much text to process at once (256-1024 typical)
- **Chunk Overlap**: Overlap between chunks to maintain context (10-100 typical)

### Similarity Top-K (1-20)
- **Lower values (3-5)**: More focused, faster responses
- **Higher values (8-15)**: More comprehensive, detailed responses

### Node Postprocessors
- **SimilarityPostprocessor**: Filters out low-relevance documents

### Similarity Cutoff (0.0-1.0)
- **0.1-0.3**: More permissive, includes potentially relevant docs
- **0.5-0.8**: More strict, only highly relevant docs

### Response Synthesizers
- **TreeSummarize**: Hierarchical summarization, good for complex topics
- **Refine**: Iterative refinement, builds detailed responses
- **CompactAndRefine**: Efficient version of Refine
- **Default**: Standard synthesis approach


## ✅ Assignment Completion Checklist

Before submitting, ensure you have:

- [ ] Set up your OPENROUTER_API_KEY environment variable
- [ ] Imported all necessary libraries including advanced RAG components
- [ ] Created AdvancedRAGBackend class with configurable parameters
- [ ] Implemented all required methods:
  - [ ] `update_settings()` - Updates LLM and chunking parameters
  - [ ] `initialize_database()` - Sets up vector database
  - [ ] `get_postprocessor()` - Returns selected postprocessor
  - [ ] `get_synthesizer()` - Returns selected synthesizer
  - [ ] `advanced_query()` - Handles queries with all configuration options
- [ ] Created advanced Gradio interface with all required components:
  - [ ] Initialize database button
  - [ ] Model selection dropdown (gpt-4o, gpt-4o-mini)
  - [ ] Temperature slider (0 to 1, step 0.1)
  - [ ] Chunk size input (default 512)
  - [ ] Chunk overlap input (default 50)
  - [ ] Similarity top-k slider (1 to 20, default 5)
  - [ ] Node postprocessor multiselect
  - [ ] Similarity cutoff slider (0.0 to 1.0, step 0.1, default 0.3)
  - [ ] Response synthesizer dropdown
  - [ ] Query input and submit button
  - [ ] Response output
  - [ ] Configuration display
- [ ] Connected all components to backend functions
- [ ] Successfully launched the application
- [ ] Tested different parameter combinations
- [ ] Verified all configuration options work correctly

## 🎊 Congratulations!

You've successfully built a professional, production-ready RAG application! You now have:

- **Advanced Parameter Control**: Full control over all RAG system parameters
- **Professional UI**: Clean, organized interface with proper layout
- **Real-time Configuration**: Ability to experiment with different settings
- **Production Patterns**: Understanding of how to build scalable AI applications

## 🚀 Next Steps & Extensions

**Potential Enhancements:**
1. **Authentication**: Add user login and session management
2. **Document Upload**: Allow users to upload their own documents
3. **Chat History**: Implement conversation memory
4. **Performance Monitoring**: Add response time and quality metrics
5. **A/B Testing**: Compare different configurations side-by-side
6. **Export Features**: Download responses and configurations
7. **Advanced Visualizations**: Show document similarity scores and retrieval paths

**Deployment Options:**
- **Local**: Run on your machine for development
- **Gradio Cloud**: Deploy with `interface.launch(share=True)`
- **Hugging Face Spaces**: Deploy to Hugging Face for public access
- **Docker**: Containerize for scalable deployment
- **Cloud Platforms**: Deploy to AWS, GCP, or Azure

You're now ready to build sophisticated AI-powered applications!
